In [13]:
import json, time, unicodedata, re
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

HG_MODEL = "livekit/turn-detector"
ONNX_FILENAME = "model_q8.onnx"
MODEL_REVISION = "v0.4.1-intl"   # multilingual. English-only bo'lsa: "v1.2.2-en"
MAX_HISTORY_TOKENS = 128
TEST_LANGUAGE = "en"   # supported: en es fr de it pt nl zh ja ko id tr ru hi

onnx_path = hf_hub_download(HG_MODEL, ONNX_FILENAME, subfolder="onnx",
                           revision=MODEL_REVISION, local_files_only=True)
lang_path = hf_hub_download(HG_MODEL, "languages.json",
                           revision=MODEL_REVISION, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(HG_MODEL, revision=MODEL_REVISION,
                                          local_files_only=True, truncation_side="left")
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

with open(lang_path) as f:
    languages = json.load(f)
threshold = languages.get(TEST_LANGUAGE, {}).get("threshold", 0.5)
print(f"Ready. language={TEST_LANGUAGE}, threshold={threshold:.4f}")

Ready. language=en, threshold=0.0110


In [6]:
def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text.lower())
    text = "".join(ch for ch in text
                   if not (unicodedata.category(ch).startswith("P") and ch not in ["'", "-"]))
    return re.sub(r"\s+", " ", text).strip()

def format_chat_ctx(chat_ctx):
    new_ctx, last = [], None
    for msg in chat_ctx:
        if not msg["content"]:
            continue
        content = normalize_text(msg["content"])
        if last and last["role"] == msg["role"]:
            last["content"] += f" {content}"
        else:
            m = {"role": msg["role"], "content": content}
            new_ctx.append(m); last = m
    convo = tokenizer.apply_chat_template(new_ctx, add_generation_prompt=False,
                                          add_special_tokens=False, tokenize=False)
    return convo[:convo.rfind("<|im_end|>")]

def predict(chat_ctx):
    """chat_ctx = [{'role':'user'/'assistant','content':...}]. Oxirgisi user bo'lishi kerak."""
    text = format_chat_ctx(chat_ctx)
    inputs = tokenizer(text, add_special_tokens=False, return_tensors="np",
                       max_length=MAX_HISTORY_TOKENS, truncation=True)
    t0 = time.perf_counter()
    out = session.run(None, {"input_ids": inputs["input_ids"].astype("int64")})
    eou = float(out[0].flatten()[-1])
    dt = (time.perf_counter() - t0) * 1000
    finished = eou >= threshold
    print(f"  '{chat_ctx[-1]['content']}'")
    print(f"  eou_probability = {eou:.4f}  (threshold {threshold:.3f}, {dt:.0f} ms)")
    print(f"  -> {'FINISHED, agent responds' if finished else 'STILL SPEAKING, agent waits'}\n")
    return eou

In [7]:
print("en threshold:", languages.get("en"))
print("hamma tillar:", list(languages.keys()))
print("ishlatilayotgan threshold:", threshold)

en threshold: {'threshold': 0.011, 'tpr': 0.993, 'tnr': 0.8696}
hamma tillar: ['zh', 'de', 'nl', 'en', 'pt', 'es', 'fr', 'it', 'ja', 'ko', 'ru', 'tr', 'id', 'hi']
ishlatilayotgan threshold: 0.011


In [8]:
# bitta gap:
predict([{"role": "user", "content": "what time is it"}])
predict([{"role": "user", "content": "i need to think about that for a moment"}])

# ko'p bosqichli kontekst bilan:
predict([
    {"role": "assistant", "content": "How can I help you today?"},
    {"role": "user", "content": "so basically my flight got cancelled and"},
])

  'what time is it'
  eou_probability = 0.0510  (threshold 0.011, 26 ms)
  -> FINISHED, agent responds

  'i need to think about that for a moment'
  eou_probability = 0.2129  (threshold 0.011, 6 ms)
  -> FINISHED, agent responds

  'so basically my flight got cancelled and'
  eou_probability = 0.0002  (threshold 0.011, 10 ms)
  -> STILL SPEAKING, agent waits



0.00018844236910808831

In [ ]:
predict([{"role": "user", "content": "Men bugun maktabga"}])
predict([{"role": "user", "content": "hmmm manimcha keladi"}])
predict([{"role": "user", "content": "yoq bilmadim lekin"}])
predict([{"role": "user", "content": "telefon raqamim 91 695"}])


  'Men bugun maktabga'
  eou_probability = 0.0830  (threshold 0.011, 11 ms)
  -> FINISHED, agent responds

  'hmmm manimcha keladi'
  eou_probability = 0.0944  (threshold 0.011, 8 ms)
  -> FINISHED, agent responds

  'yoq bilmadim lekin'
  eou_probability = 0.0424  (threshold 0.011, 8 ms)
  -> FINISHED, agent responds

  'telefon raqamim 91 695'
  eou_probability = 0.0034  (threshold 0.011, 7 ms)
  -> STILL SPEAKING, agent waits

  'soat nechchi bo'ldi'
  eou_probability = 0.3235  (threshold 0.011, 7 ms)
  -> FINISHED, agent responds

  'menga bir daqiqa o'ylab ko'ray'
  eou_probability = 0.0681  (threshold 0.011, 6 ms)
  -> FINISHED, agent responds

  'gap shundaki'
  eou_probability = 0.2252  (threshold 0.011, 5 ms)
  -> FINISHED, agent responds

  'men uchishimni bekor qilishdi va'
  eou_probability = 0.0138  (threshold 0.011, 7 ms)
  -> FINISHED, agent responds

  'aytmoqchi edimki'
  eou_probability = 0.2081  (threshold 0.011, 7 ms)
  -> FINISHED, agent responds



0.20813797414302826

In [12]:
predict([{"role": "user", "content": "soat nechchi bo'ldi"}])            # tugagan
predict([{"role": "user", "content": "menga bir daqiqa o'ylab ko'ray"}]) # tugagan
predict([{"role": "user", "content": "gap shundaki"}])                   # tugallanmagan
predict([{"role": "user", "content": "men uchishimni bekor qilishdi va"}]) # tugallanmagan
predict([{"role": "user", "content": "aytmoqchi edimki"}])               # tugallanmagan

  'soat nechchi bo'ldi'
  eou_probability = 0.3235  (threshold 0.011, 12 ms)
  -> FINISHED, agent responds

  'menga bir daqiqa o'ylab ko'ray'
  eou_probability = 0.0681  (threshold 0.011, 9 ms)
  -> FINISHED, agent responds

  'gap shundaki'
  eou_probability = 0.2252  (threshold 0.011, 6 ms)
  -> FINISHED, agent responds

  'men uchishimni bekor qilishdi va'
  eou_probability = 0.0138  (threshold 0.011, 8 ms)
  -> FINISHED, agent responds

  'aytmoqchi edimki'
  eou_probability = 0.2081  (threshold 0.011, 6 ms)
  -> FINISHED, agent responds



0.20813797414302826